# Threat Hunting Notebook

Hunts against the Wazuh indexer using OpenSearch DSL queries. Each cell
represents one hypothesis-driven hunt aligned to a MITRE ATT&CK technique.

**Connection:** the notebook expects the SOC lab stack to be running with
the indexer reachable at `https://localhost:9200` and credentials
`admin / SecretPassword`.

**Hunts included:**
1. Rare process parent-child relationships (T1059)
2. First-seen external IPs by user (T1078)
3. Off-hours admin activity (T1078.002)
4. PowerShell encoded command frequency (T1027)
5. DNS queries to high-entropy domains (T1568.002)
6. SMB connections from non-domain hosts (T1021.002)

In [ ]:
import math
import warnings
from datetime import datetime, timedelta

import pandas as pd
from opensearchpy import OpenSearch

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 80)

client = OpenSearch(
    hosts=[{'host': 'localhost', 'port': 9200}],
    http_auth=('admin', 'SecretPassword'),
    use_ssl=True,
    verify_certs=False,
    ssl_show_warn=False,
)
INDEX = 'wazuh-alerts-*'
print('Connected:', client.info()['version']['distribution'])

## Hunt 1: Rare process parent-child relationships

**Hypothesis:** legitimate software has a stable set of parent-child process
patterns. Rare combinations are worth investigating (e.g. `winword.exe` spawning
`powershell.exe` is a classic Office macro red flag).

In [ ]:
query = {
    'size': 0,
    'query': {'bool': {'filter': [
        {'range': {'@timestamp': {'gte': 'now-7d'}}},
        {'exists': {'field': 'data.win.eventdata.parentImage'}},
    ]}},
    'aggs': {
        'parent_child': {
            'composite': {
                'size': 1000,
                'sources': [
                    {'parent': {'terms': {'field': 'data.win.eventdata.parentImage.keyword'}}},
                    {'child': {'terms': {'field': 'data.win.eventdata.image.keyword'}}},
                ],
            }
        }
    },
}
r = client.search(index=INDEX, body=query)
buckets = r['aggregations']['parent_child']['buckets']
df = pd.DataFrame([
    {'parent': b['key']['parent'], 'child': b['key']['child'], 'count': b['doc_count']}
    for b in buckets
])
rare = df[df['count'] <= 3].sort_values('count')
rare.head(20)

## Hunt 2: First-seen external IPs by user

**Hypothesis:** users typically authenticate from a stable set of source IPs.
A new IP for an existing user could indicate credential theft.

In [ ]:
query = {
    'size': 0,
    'query': {'bool': {'must': [
        {'match': {'rule.groups': 'authentication_success'}},
        {'range': {'@timestamp': {'gte': 'now-30d'}}},
    ]}},
    'aggs': {
        'users': {
            'terms': {'field': 'data.srcuser.keyword', 'size': 100},
            'aggs': {'ips': {'terms': {'field': 'data.srcip.keyword', 'size': 50}}},
        }
    },
}
r = client.search(index=INDEX, body=query)
rows = []
for u in r['aggregations']['users']['buckets']:
    for ip in u['ips']['buckets']:
        rows.append({'user': u['key'], 'src_ip': ip['key'], 'count': ip['doc_count']})
df = pd.DataFrame(rows)
first_seen = df.groupby('user').filter(lambda g: len(g) > 1)
first_seen.sort_values(['user', 'count']).head(30)

## Hunt 3: Off-hours admin activity

**Hypothesis:** privileged commands at 3 AM are statistically unusual.

In [ ]:
query = {
    'size': 100,
    'query': {'bool': {'must': [
        {'match': {'rule.groups': 'sudo'}},
        {'range': {'@timestamp': {'gte': 'now-7d'}}},
    ]}},
    'sort': [{'@timestamp': 'desc'}],
}
r = client.search(index=INDEX, body=query)
df = pd.DataFrame([{
    'ts': h['_source']['@timestamp'],
    'user': h['_source'].get('data', {}).get('srcuser'),
    'cmd': h['_source'].get('full_log', '')[:80],
} for h in r['hits']['hits']])
df['hour'] = pd.to_datetime(df['ts']).dt.hour
df[(df['hour'] < 6) | (df['hour'] > 22)]

## Hunt 4: High-entropy DNS subdomains

**Hypothesis:** DGAs and DNS tunneling produce subdomains with abnormally
high Shannon entropy.

In [ ]:
def shannon(s):
    if not s:
        return 0.0
    probs = [s.count(c) / len(s) for c in set(s)]
    return -sum(p * math.log2(p) for p in probs)

query = {
    'size': 1000,
    'query': {'bool': {'must': [
        {'exists': {'field': 'data.dns.query'}},
        {'range': {'@timestamp': {'gte': 'now-1d'}}},
    ]}},
}
r = client.search(index=INDEX, body=query)
rows = []
for h in r['hits']['hits']:
    q = h['_source'].get('data', {}).get('dns', {}).get('query', '')
    sub = q.split('.')[0] if '.' in q else q
    if len(sub) > 8:
        rows.append({'query': q, 'subdomain': sub, 'entropy': shannon(sub)})
df = pd.DataFrame(rows).sort_values('entropy', ascending=False)
df.head(20)